# 10 · Viés de gênero em português: gênero gramatical, profissões epicenas e um experimento controlado

> **Disciplina:** Representações de Linguagem — *como transformar palavra em vetor, o que esse vetor carrega e o viés que vem junto* (BLIS 2S26).
> Este notebook faz parte de uma sequência (`00` → `13`). Código de apoio em `src/replang`; interface interativa em `app/` (`uv run replang app`).


Bolukbasi et al. (2016) terminam com uma pergunta: *"it is also an interesting direction to consider how the approach and findings here would apply to other languages, especially languages with grammatical gender where the definitions of most nouns carry a gender marker"* (§9). O português é exatamente esse caso. Este notebook:

1. aplica o método ao **Wikipedia2Vec PT** e ao **Machado de Assis** (direção de gênero, extremos, DirectBias);
2. discute o problema do **gênero gramatical**: *enfermeira/enfermeiro* carregam gênero na ortografia — isso é viés? (não: é morfologia) — e por isso usa **profissões epicenas** (*dentista, gerente, cientista*) e **áreas** (*enfermagem, engenharia*);
3. separa, nas analogias geradas, pares **morfológicos** de **estereótipos**;
4. faz um **experimento controlado** com corpus sintético, variando o viés do corpus de 50 % a 100 % e medindo o viés geométrico resultante;
5. aplica o hard debias em PT e discute o que ele faz com o gênero gramatical.

In [1]:
# Configuração comum a todos os notebooks ---------------------------------------------
import sys, warnings, os
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"  # JupyterLab renderiza offline; HTML carrega plotly.js via CDN (notebooks leves)
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 160)
from replang import PATHS, WordVectors, settings
FAST = settings.fast  # REPLANG_FAST=1 reduz épocas/tamanhos
print("raiz:", ROOT, "| fast:", FAST)

raiz: /home/user/linguagem-blis2s26 | fast: False


In [2]:
from replang.data.embeddings import load_ptwiki, load_local_model
from replang.data.lexicons import PT
from replang.bias import gender_direction, pca_pairs, random_pca_baseline, project, extremes, direct_bias, indirect_bias, generate_analogies, split_analogies_pt, hard_debias, pair_bias
from replang.bias.geometry import profile
from replang.viz import pca_variance_figure, bias_axis_figure
lex = PT.get()
pt = load_ptwiki().normalized("Wikipedia2Vec PT 100d"); mach = load_local_model("machado_sg100").normalized("Machado SG 100d")
print(pt, "|", mach)
print("pares definicionais PT:", lex.definitional_pairs)
print(f"profissões epicenas: {len(lex.professions)} | áreas/substantivos neutros: {len(lex.stereotype_neutral)} | pares morfológicos de profissão: {len(lex.professions_gendered_pairs)}")

WordVectors(name='Wikipedia2Vec PT 100d', n=50000, dim=100) | WordVectors(name='Machado SG 100d', n=23194, dim=100)
pares definicionais PT: [('mulher', 'homem'), ('ela', 'ele'), ('menina', 'menino'), ('mãe', 'pai'), ('filha', 'filho'), ('irmã', 'irmão'), ('rainha', 'rei'), ('esposa', 'marido'), ('feminino', 'masculino'), ('maria', 'joão')]
profissões epicenas: 65 | áreas/substantivos neutros: 68 | pares morfológicos de profissão: 31


## 1. A direção de gênero em PT (Fig. 6, versão PT)

Em português a PCA das diferenças dos pares é ainda mais "limpa" que em inglês: a 1ª componente explica mais de 60 % da variância no Wikipedia2Vec. Hipótese: o gênero gramatical faz *toda* a concordância (artigos, adjetivos, particípios) coocorrer com o gênero, reforçando a direção.

In [3]:
for wv in [pt, mach]:
    comps, ev, used = pca_pairs(wv, lex.definitional_pairs)
    g = gender_direction(wv, lex.definitional_pairs)
    print(f"{wv.name}: pares usados {len(used)} · variância explicada {np.round(ev[:4], 3)} · aleatório {random_pca_baseline(wv.dim, len(used))[0]:.3f} · cos(ela,g)={float(wv['ela'] @ g):+.2f} cos(ele,g)={float(wv['ele'] @ g):+.2f}")
g_pt = gender_direction(pt, lex.definitional_pairs); g_m = gender_direction(mach, lex.definitional_pairs)
_, ev_pt, used_pt = pca_pairs(pt, lex.definitional_pairs)
pca_variance_figure(ev_pt, random_pca_baseline(pt.dim, len(used_pt)), title="PT (Wikipedia2Vec): variância explicada pela PCA das diferenças dos pares × aleatório")

Wikipedia2Vec PT 100d: pares usados 10 · variância explicada [0.618 0.121 0.072 0.057] · aleatório 0.155 · cos(ela,g)=+0.26 cos(ele,g)=-0.12
Machado SG 100d: pares usados 10 · variância explicada [0.413 0.168 0.11  0.076] · aleatório 0.155 · cos(ela,g)=+0.29 cos(ele,g)=+0.02


## 2. O que está nos extremos do eixo *ela–ele*

Projetamos três grupos: (a) **profissões epicenas** (forma única para os dois gêneros), (b) **áreas e substantivos abstratos** (*cozinha, engenharia, coragem, delicadeza*) e (c) **pares morfológicos** (*enfermeira/enfermeiro*). Em (a) e (b) a ortografia não explica a posição; em (c) explica — e esse é o ponto da discussão.

In [4]:
epicenas = [w for w in lex.professions if w in pt]; areas = [w for w in lex.stereotype_neutral if w in pt]
fem, masc = extremes(pt, g_pt, epicenas + areas, 15)
print("extremo ELA:", fem); print("extremo ELE:", masc)
print(f"\nDirectBias_1 (Wikipedia2Vec PT): epicenas = {direct_bias(pt, epicenas, g_pt):.3f} | áreas = {direct_bias(pt, areas, g_pt):.3f} | 300 palavras frequentes não-gênero = {direct_bias(pt, [w for w in pt.words[:600] if w not in lex.all_gender_words()][:300], g_pt):.3f}")
print(f"DirectBias_1 (Machado SG):        epicenas = {direct_bias(mach, [w for w in epicenas if w in mach], g_m):.3f} | áreas = {direct_bias(mach, [w for w in areas if w in mach], g_m):.3f}")

extremo ELA: ['beleza', 'babá', 'dança', 'recepcionista', 'moda', 'costura', 'cozinha', 'casa', 'criança', 'música', 'maquiagem', 'delicadeza', 'ternura', 'sensibilidade', 'escola']
extremo ELE: ['sábio', 'capitão', 'gênio', 'general', 'líder', 'escritor', 'comerciante', 'vilão', 'herói', 'poder', 'juiz', 'soldado', 'presidente', 'dirigente', 'boxe']

DirectBias_1 (Wikipedia2Vec PT): epicenas = 0.117 | áreas = 0.109 | 300 palavras frequentes não-gênero = 0.064
DirectBias_1 (Machado SG):        epicenas = 0.150 | áreas = 0.117


In [5]:
df = profile(pt, epicenas + areas, g_pt, lex.extra["labels"])
sel = pd.concat([df.nsmallest(20, "proj_g"), df.nlargest(20, "proj_g")])
bias_axis_figure(sel, title="Profissões epicenas e áreas projetadas na direção de gênero (Wikipedia2Vec PT)", axis_label="← ele          ela →")

In [6]:
# Pares morfológicos: os dois membros se separam em g (esperado — gênero gramatical), mas o CENTRO do par também pode estar deslocado (estereótipo)
rows = []
for f, m in lex.professions_gendered_pairs:
    if pt.has(f, m):
        pf, pm = float(pt[f] @ g_pt), float(pt[m] @ g_pt)
        rows.append({"par": f"{f}/{m}", "proj. feminina": pf, "proj. masculina": pm, "separação (gramatical)": pf - pm, "centro do par (estereótipo?)": (pf + pm) / 2})
pairs_df = pd.DataFrame(rows).sort_values("centro do par (estereótipo?)")
pairs_df.style.format({c: "{:+.3f}" for c in pairs_df.columns[1:]}).background_gradient(subset=["centro do par (estereótipo?)"], cmap="RdBu_r")

,par,proj. feminina,proj. masculina,separação (gramatical),centro do par (estereótipo?)
25,mecânica/mecânico,-0.020,-0.187,+0.167,-0.103
24,pedreira/pedreiro,+0.028,-0.190,+0.218,-0.081
22,operária/operário,+0.073,-0.212,+0.285,-0.069
19,física/físico,+0.051,-0.157,+0.208,-0.053
20,matemática/matemático,+0.084,-0.181,+0.265,-0.048
21,química/químico,+0.067,-0.150,+0.218,-0.041
23,camponesa/camponês,+0.152,-0.204,+0.356,-0.026
1,engenheira/engenheiro,+0.298,-0.269,+0.567,+0.015
3,médica/médico,+0.178,-0.136,+0.314,+0.021
12,vereadora/vereador,+0.213,-0.145,+0.359,+0.034


Leitura: a coluna **separação** é o gênero gramatical (sempre positiva: a forma feminina está do lado *ela*); a coluna **centro** diz se o *conceito* da profissão, independentemente da forma, pende para um lado. *Enfermeira/enfermeiro*, *secretária/secretário*, *cozinheira/cozinheiro* tendem a centro positivo; *engenheira/engenheiro*, *físico/física*, *pedreiro/pedreira* a centro negativo. É uma maneira de **separar morfologia de estereótipo** usando a própria geometria.

## 3. Viés indireto em PT

Análogo a *softball–football*: tomamos **cozinha–futebol** e medimos quanto da similaridade com outras palavras neutras vem da direção de gênero.

In [7]:
axis = pt["cozinha"] - pt["futebol"]
coz, fut = extremes(pt, axis, epicenas + areas, 8)
rows = [{"w": "cozinha", "v": v, "cos": pt.similarity("cozinha", v), "β(w,v)": indirect_bias(pt, "cozinha", v, g_pt)} for v in coz[:6]]
rows += [{"w": "futebol", "v": v, "cos": pt.similarity("futebol", v), "β(w,v)": indirect_bias(pt, "futebol", v, g_pt)} for v in fut[:6]]
pd.DataFrame(rows).style.format({"cos": "{:.2f}", "β(w,v)": "{:+.0%}"})

,w,v,cos,"β(w,v)"
0,cozinha,cozinha,1.00,+0%
1,cozinha,atendente,0.57,-3%
2,cozinha,chef,0.67,-1%
3,cozinha,costura,0.58,+3%
4,cozinha,fábrica,0.59,+2%
5,cozinha,casa,0.63,+2%
6,futebol,futebol,1.00,+0%
7,futebol,esporte,0.67,-0%
8,futebol,vôlei,0.70,-0%
9,futebol,atleta,0.60,+0%


In [8]:
# Versão orientada a dados: pares de palavras neutras (epicenas + áreas) cuja similaridade mais depende de g
from replang.bias import top_indirect_bias_pairs
top_indirect_bias_pairs(pt, epicenas + areas, g_pt, topn=15, min_cos=0.3).style.format({"cos": "{:.2f}", "β": "{:.0%}", "proj_g(w)": "{:+.2f}", "proj_g(v)": "{:+.2f}"})

,w,v,cos,β,proj_g(w),proj_g(v)
0,general,sábio,0.33,15%,-0.23,-0.29
1,babá,beleza,0.38,12%,+0.26,+0.27
2,presidente,sábio,0.31,12%,-0.18,-0.29
3,general,gênio,0.35,11%,-0.23,-0.23
4,babá,música,0.33,10%,+0.26,+0.18
5,capitão,sábio,0.42,10%,-0.23,-0.29
6,beleza,escola,0.31,10%,+0.27,+0.16
7,babá,delicadeza,0.32,10%,+0.26,+0.18
8,comerciante,vilão,0.34,10%,-0.22,-0.22
9,recepcionista,beleza,0.39,9%,+0.22,+0.27


## 4. Analogias geradas: separando gramática de estereótipo

Com a semente (*ela*, *ele*) a eq. (1) de Bolukbasi devolve, em PT, sobretudo pares **morfológicos** (*escritora–escritor*) — corretos e esperados. A função `split_analogies_pt` usa uma heurística ortográfica para separá-los dos pares **sem relação morfológica**, que são os candidatos a estereótipo.

In [9]:
an = generate_analogies(pt, "ela", "ele", delta=1.0, topn=120, restrict=30000, exclude=lex.all_gender_words())
gram, stereo = split_analogies_pt(an)
print(f"{len(an)} analogias geradas → {len(gram)} gramaticais, {len(stereo)} sem relação morfológica")
print("gramaticais (amostra):", list(zip(gram.x[:10], gram.y[:10])))
print("candidatos a estereótipo:")
stereo.head(25).T

120 analogias geradas → 77 gramaticais, 43 sem relação morfológica
gramaticais (amostra): [('nascida', 'nascido'), ('sozinha', 'sozinho'), ('historiadora', 'historiador'), ('candidata', 'candidato'), ('pesquisadora', 'pesquisador'), ('substituta', 'substituto'), ('convidada', 'convidado'), ('dubladora', 'dublador'), ('autora', 'autor'), ('defensora', 'defensor')]
candidatos a estereótipo:


,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
x,talentosa,carmem,campeã,vilã,apresentadora,compositora,loira,sensual,catherine,linda,...,susan,governanta,baronesa,teodora,laura,fotógrafa,alessandra,júlia,novata,bicampeã
y,habilidoso,valdir,campeão,vilão,comentarista,músico,bigode,vigoroso,bernard,emerson,...,gerald,ajudante,barão,basílio,david,ilustrador,márcio,saturnino,experiente,bicampeão
score,0.777735,0.728289,0.703459,0.689032,0.684443,0.676015,0.675885,0.673657,0.667715,0.667402,...,0.657496,0.656671,0.6566,0.655744,0.653376,0.652269,0.651866,0.651586,0.648952,0.643099
dist,0.854455,0.899819,0.539428,0.726342,0.840873,0.793517,0.866295,0.916174,0.770841,0.996436,...,0.699952,0.933337,0.674975,0.716413,0.810823,0.913174,0.79685,0.84984,0.90309,0.538152


In [10]:
# Com uma semente "conceitual" em vez de pronominal: mulher–homem, mãe–pai
for a, b in [("mulher", "homem"), ("mãe", "pai")]:
    an2 = generate_analogies(pt, a, b, delta=1.0, topn=60, restrict=30000, exclude=lex.all_gender_words())
    _, st2 = split_analogies_pt(an2)
    print(f"{a}:{b} → candidatos a estereótipo: {list(zip(st2.x[:12], st2.y[:12]))}")

mulher:homem → candidatos a estereótipo: [('vilã', 'vilão'), ('poetisa', 'poeta'), ('baronesa', 'barão'), ('historiadora', 'estudioso'), ('governanta', 'jardineiro'), ('prostituta', 'ladrão'), ('aluna', 'discípulo'), ('socialite', 'excêntrico'), ('ellen', 'alvin'), ('diva', 'cowboy'), ('assessora', 'consultor'), ('suzanne', 'gary')]


mãe:pai → candidatos a estereótipo: [('poetisa', 'poeta'), ('vilã', 'vilão'), ('ana', 'alberto'), ('infanta', 'primogénito'), ('baronesa', 'barão'), ('padroeira', 'patrono'), ('dra', 'dr'), ('bruxa', 'mago'), ('sra', 'sr'), ('barbara', 'robert'), ('eleanor', 'henry'), ('consorte', 'herdeiro')]


## 5. Experimento controlado: o viés geométrico reflete o viés do corpus?

Bolukbasi et al. argumentam que o viés vem do texto. Testamos **causalmente** com um corpus sintético (`synthetic_gender_corpus`): frases-molde em que profissões "estereotipadas" coocorrem com o gênero esperado com probabilidade `bias` ∈ {0,5; 0,6; …; 1,0} e profissões "neutras" com 50 %. Treinamos um Skip-gram (numpy) para cada valor e medimos a projeção das profissões na direção de gênero. Se a hipótese está certa, o DirectBias cresce com `bias` **e** as neutras ficam em zero.

In [11]:
import time
from replang.data.corpora import synthetic_gender_corpus, SYNTHETIC_PROFESSIONS
from replang.models.word2vec_np import Word2Vec
from replang.accel import detect
try:
    from replang.models.word2vec_jax import SkipGramJax
    USE_JAX = detect().use_jax_for("sgns")
except Exception:
    USE_JAX = False
print("backend do Skip-gram:", "JAX (jit + scatter-add; GPU se disponível)" if USE_JAX else "numpy")
pairs_syn = [("ela", "ele"), ("mulher", "homem"), ("mãe", "pai"), ("filha", "filho"), ("irmã", "irmão"), ("menina", "menino"), ("rainha", "rei"), ("esposa", "marido")]
rows, profiles = [], {}
for bias in ([0.5, 0.75, 1.0] if FAST else [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]):
    t = time.time()
    corpus = synthetic_gender_corpus(15000, bias=bias, seed=0)
    # sem subamostragem: no corpus sintético as palavras de gênero são frequentíssimas e seriam descartadas (t=1e-3 apaga o sinal!)
    if USE_JAX:
        m = SkipGramJax(dim=50, window=5, negative=10, epochs=8, sample=0, seed=0).train(corpus)
    else:
        m = Word2Vec(dim=50, window=5, negative=10, epochs=8, sample=0, batch_size=128, seed=0).train(corpus)
    wv = m.to_wordvectors().normalized()
    g = gender_direction(wv, pairs_syn)
    pf = project(wv, SYNTHETIC_PROFESSIONS["estereotipo_feminino"], g); pm = project(wv, SYNTHETIC_PROFESSIONS["estereotipo_masculino"], g); pn = project(wv, SYNTHETIC_PROFESSIONS["neutras"], g)
    profiles[bias] = (pf, pm, pn)
    rows.append({"viés do corpus": bias, "média proj. estereótipo fem.": pf.mean(), "média proj. estereótipo masc.": pm.mean(), "separação fem − masc": pf.mean() - pm.mean(), "média |proj.| neutras": pn.abs().mean(),
                 "DirectBias (estereotipadas)": direct_bias(wv, list(pf.index) + list(pm.index), g), "DirectBias (neutras)": direct_bias(wv, list(pn.index), g), "s": round(time.time() - t)})
syn_df = pd.DataFrame(rows)
syn_df.style.format({c: "{:+.3f}" for c in syn_df.columns[1:-1]})

backend do Skip-gram: JAX (jit + scatter-add; GPU se disponível)


,viés do corpus,média proj. estereótipo fem.,média proj. estereótipo masc.,separação fem − masc,média |proj.| neutras,DirectBias (estereotipadas),DirectBias (neutras),s
0,0.500000,+0.048,+0.024,+0.024,+0.163,+0.132,+0.163,19
1,0.600000,+0.261,-0.289,+0.550,+0.052,+0.275,+0.052,19
2,0.700000,+0.356,-0.344,+0.700,+0.025,+0.350,+0.025,19
3,0.800000,+0.386,-0.378,+0.764,+0.022,+0.382,+0.022,19
4,0.900000,+0.420,-0.405,+0.826,+0.020,+0.413,+0.020,18
5,1.000000,+0.442,-0.423,+0.865,+0.036,+0.432,+0.036,19


In [12]:
import plotly.express as px
m = syn_df.melt(id_vars="viés do corpus", value_vars=["separação fem − masc", "DirectBias (estereotipadas)", "DirectBias (neutras)"], var_name="métrica", value_name="valor")
px.line(m, x="viés do corpus", y="valor", color="métrica", markers=True, template="plotly_white", title="Viés geométrico cresce com o viés estatístico do corpus; profissões neutras ficam perto de zero").show()

Com `bias = 0.5` (corpus "justo") a direção de gênero **ainda existe** (os pares definicionais a criam), mas as profissões estereotipadas e as neutras ficam todas perto de zero (ruído). A partir de `0.7` a separação fem − masc dispara e satura perto de `1.0`. A geometria é um **termômetro** das coocorrências: nenhum algoritmo "inventa" o estereótipo, ele o **mede e reproduz**.

Detalhe técnico que vale um comentário em sala: a **subamostragem** (`sample`) precisou ser desligada — no corpus sintético as palavras de gênero são tão frequentes que `t = 10⁻³` descartaria quase todas as suas ocorrências e apagaria o sinal. Hiperparâmetros "inocentes" mudam o que o embedding carrega.

## 6. Hard debias em português

Aplicamos Neutralize + Equalize ao Wikipedia2Vec PT com $N$ = tudo menos as palavras específicas de gênero e $E$ = pares de equalização PT (inclui pares morfológicos de parentesco e títulos). **Decisão de projeto**: os pares morfológicos de **profissão** (*enfermeira/enfermeiro*) entram como conjuntos de igualdade? Se sim, *enfermeira* e *enfermeiro* ficam simétricos e o conceito "enfermagem" é neutralizado; se não, eles são tratados como neutras e **perdem** a marca de gênero — o que destruiria concordância em aplicações gerativas. Mostramos as duas opções.

In [13]:
B = g_pt[None, :]
opt_a = hard_debias(pt, B, gender_specific=lex.gender_specific, equalize_pairs=lex.equalize_pairs + lex.professions_gendered_pairs, name="PT hard (profissões equalizadas)")
spec_b = [w for w in lex.gender_specific if w not in {x for p in lex.professions_gendered_pairs for x in p}]
opt_b = hard_debias(pt, B, gender_specific=spec_b, equalize_pairs=lex.equalize_pairs, name="PT hard (profissões neutralizadas)")
neutras = epicenas + areas
rows = []
for wv in [pt, opt_a, opt_b]:
    rows.append({"embedding": wv.name, "DirectBias epicenas+áreas": direct_bias(wv, neutras, g_pt), "PairBias": pair_bias(wv, neutras, lex.definitional_pairs),
                 "cos(enfermeira,g)": float(wv["enfermeira"] @ g_pt), "cos(enfermeiro,g)": float(wv["enfermeiro"] @ g_pt), "cos(enfermeira,enfermeiro)": wv.similarity("enfermeira", "enfermeiro"), "cos(rei,rainha)": wv.similarity("rei", "rainha")})
pd.DataFrame(rows).style.format({c: "{:.3f}" for c in rows[0] if c != "embedding"})

,embedding,DirectBias epicenas+áreas,PairBias,"cos(enfermeira,g)","cos(enfermeiro,g)","cos(enfermeira,enfermeiro)","cos(rei,rainha)"
0,Wikipedia2Vec PT 100d,0.113,0.063,0.400,-0.070,0.699,0.732
1,PT hard (profissões equalizadas),0.018,0.012,0.422,-0.422,0.644,0.725
2,PT hard (profissões neutralizadas),0.000,0.000,-0.000,0.000,0.795,0.725


In [14]:
for a, b, c in [("ele", "médico", "ela"), ("homem", "programador", "mulher"), ("ela", "enfermeira", "ele"), ("ele", "engenheiro", "ela")]:
    print(f"{a}:{b} :: {c}:?")
    for wv in [pt, opt_a]:
        print(f"   {wv.name:<38} {[x for x, _ in wv.analogy(a, b, c, topn=5, exclude_words=lex.gender_specific)]}")
print("\nvizinhos de 'enfermagem' antes :", [x for x, _ in pt.most_similar("enfermagem", topn=8)])
print("vizinhos de 'enfermagem' depois:", [x for x, _ in opt_a.most_similar("enfermagem", topn=8)])

ele:médico :: ela:?
   Wikipedia2Vec PT 100d                  ['pediatra', 'psiquiatra', 'médica', 'dentista', 'cirurgião']
   PT hard (profissões equalizadas)       ['médica', 'veterinário', 'obstetrícia', 'pediatra', 'dentista']
homem:programador :: mulher:?
   Wikipedia2Vec PT 100d                  ['desenvolvedor', 'musicista', 'diretora', 'fotógrafa', 'coreógrafa']
   PT hard (profissões equalizadas)       ['desenvolvedor', 'engenheira', 'diretora', 'programar', 'hardware']
ela:enfermeira :: ele:?
   Wikipedia2Vec PT 100d                  ['enfermeiro', 'médico', 'cirurgião', 'dentista', 'pediatra']
   PT hard (profissões equalizadas)       ['enfermeiro', 'dentista', 'cirurgião', 'pediatra', 'veterinário']
ele:engenheiro :: ela:?
   Wikipedia2Vec PT 100d                  ['engenheira', 'eletricista', 'arquiteta', 'engenharia', 'agrônomo']
   PT hard (profissões equalizadas)       ['engenheira', 'eletricista', 'agrônomo', 'engenharia', 'projetista']

vizinhos de 'enfermagem' antes 

## 7. E no corpus literário? (Machado de Assis)

Um corpus do século XIX tem uma estrutura de gênero muito marcada (papéis sociais, vocabulário de salão). O eixo *ela–ele* do modelo Machado mostra isso de forma crua — e lembra que o "viés" depende do corpus **e da época**.

In [15]:
cands = [w for w in mach.words[:6000] if w not in lex.all_gender_words() and w.isalpha() and len(w) > 3]
fem_m, masc_m = extremes(mach, g_m, cands, 25)
print("lado ELA (Machado):", fem_m); print("lado ELE (Machado):", masc_m)

lado ELA (Machado): ['amiga', 'meiga', 'prima', 'terna', 'confissão', 'ternura', 'isabel', 'criatura', 'velha', 'fria', 'dona', 'sincera', 'fronte', 'ventura', 'beleza', 'deliciosa', 'bondade', 'morta', 'formosa', 'melancólica', 'querida', 'vênus', 'bela', 'linda', 'graciosa']
lado ELE (Machado): ['brasileiro', 'processo', 'vasto', 'banco', 'número', 'navio', 'jornal', 'conservatório', 'político', 'volume', 'comércio', 'ofício', 'moderno', 'partido', 'povo', 'método', 'grosso', 'magistrado', 'pequeno', 'general', 'gênero', 'publicado', 'artigo', 'orador', 'impresso']


## 8. Síntese

1. O método de Bolukbasi **transfere** para o português: a direção existe (até mais nítida) e os estereótipos ocupacionais aparecem nas profissões epicenas e nas áreas.
2. **Gênero gramatical ≠ viés**: é preciso separar *separação dentro do par* (morfologia) de *deslocamento do centro do par* (estereótipo), e filtrar analogias morfológicas.
3. O experimento sintético confirma a relação causal corpus → geometria, com um "falso positivo" a evitar: a direção existe mesmo sem viés nas profissões.
4. O *debias* em PT exige decidir **o que preservar**: concordância gramatical é informação, não preconceito.

### Perguntas para discussão
* Neutralizar *enfermeira* destrói a concordância (*a enfermeira competente*). Como um modelo gerativo em PT deveria lidar com isso? (gancho: modelos contextuais e *debias* em nível de tarefa)
* Que outras direções valeria medir em PT? (regional, racial, de classe — Bolukbasi §9; e com que pares definicionais?)

## Referências
- BOLUKBASI, T.; CHANG, K.-W.; ZOU, J.; SALIGRAMA, V.; KALAI, A. **Man is to Computer Programmer as Woman is to Homemaker? Debiasing Word Embeddings.** NIPS, 2016. [arXiv:1607.06520](https://arxiv.org/abs/1607.06520)
- HARTMANN, N. et al. **Portuguese Word Embeddings: Evaluating on Word Analogies and Natural Language Tasks.** STIL, 2017. [arXiv:1708.06025](https://arxiv.org/abs/1708.06025)
- GONEN, H.; GOLDBERG, Y. **Lipstick on a Pig.** NAACL, 2019.